In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vit_timesformer_probe import TimeSformer, TimeSformerIBCALinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-08 20:48:43,004]::3493104939::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)

In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# TimeSformer Video-only Block-wise Linear Probe
# ============================================================
#
# Probe targets:
#   block1 ... block12 : CLS token after each complete TimeSformer block
#   final_norm         : CLS token after final LayerNorm
#
# Backbone is completely frozen.
# Only independent Linear(768, 5) probe heads are optimized.
# ============================================================

PROBE_SEED = 42
PROBE_LR = 1e-3
WEIGHT_DECAY = 1e-4
PROBE_EPOCHS = 50
PATIENCE = 10

BACKBONE_CHECKPOINT = "./save_timesformer_folder/model_best_multiseed.pth"


def set_probe_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def probe_seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_probe_seed(PROBE_SEED)

generator = torch.Generator()
generator.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

# ------------------------------------------------------------
# 1. Re-create EXACTLY the same TimeSformer backbone
#    used by the supplied original notebook.
# ------------------------------------------------------------
backbone = TimeSformer().to(device)

checkpoint = torch.load(
    BACKBONE_CHECKPOINT,
    map_location=device
)

state_dict = (
    checkpoint["model"]
    if isinstance(checkpoint, dict) and "model" in checkpoint
    else checkpoint
)

backbone.load_state_dict(
    state_dict,
    strict=True
)

# ------------------------------------------------------------
# 2. Frozen backbone + independent linear heads
# ------------------------------------------------------------
model = TimeSformerIBCALinearProbe(
    backbone=backbone,
    dim=768,
    num_outputs=5
).to(device)

backbone_trainable = sum(
    p.numel()
    for p in model.backbone.parameters()
    if p.requires_grad
)

probe_trainable = sum(
    p.numel()
    for p in model.probe_heads.parameters()
    if p.requires_grad
)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")

assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.L1Loss().to(device)

optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=PROBE_LR,
    weight_decay=WEIGHT_DECAY
)

probe_names = list(model.probe_heads.keys())

best_mae = {
    name: float("inf")
    for name in probe_names
}

best_epoch = {
    name: -1
    for name in probe_names
}

best_mean_val_mae = float("inf")
no_improve_count = 0

for epoch in range(PROBE_EPOCHS):

    # ========================================================
    # Train probe heads only
    # ========================================================
    model.train()

    train_abs_sum = {
        name: 0.0
        for name in probe_names
    }
    train_count = 0

    for audio, fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{PROBE_EPOCHS}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)

        # Preserve the original label layout while avoiding accidental
        # batch-dimension removal when batch size == 1.
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
        audio = audio.to(device)
        optimizer.zero_grad()

        outputs = model(fullshot, audio)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }

        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        scalar_count = big_five_data.numel()
        train_count += scalar_count

        for name in probe_names:
            train_abs_sum[name] += torch.abs(
                outputs[name].detach() - big_five_data
            ).sum().item()

    train_mae = {
        name: train_abs_sum[name] / train_count
        for name in probe_names
    }

    # ========================================================
    # Validation
    # ========================================================
    model.eval()

    val_abs_sum = {
        name: 0.0
        for name in probe_names
    }
    val_count = 0

    with torch.no_grad():

        for audio, fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{PROBE_EPOCHS}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
            audio = audio.to(device)
            outputs = model(fullshot, audio)

            val_count += big_five_data.numel()

            for name in probe_names:
                val_abs_sum[name] += torch.abs(
                    outputs[name] - big_five_data
                ).sum().item()

    val_mae = {
        name: val_abs_sum[name] / val_count
        for name in probe_names
    }

    # Each probe independently tracks its own best epoch.
    for name in probe_names:
        if val_mae[name] < best_mae[name]:
            best_mae[name] = val_mae[name]
            best_epoch[name] = epoch + 1

    # Global early stopping criterion = mean validation MAE
    # over all independent probe heads.
    mean_val_mae = float(
        np.mean([val_mae[name] for name in probe_names])
    )

    if mean_val_mae < best_mean_val_mae:
        best_mean_val_mae = mean_val_mae
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 90)
    print(
        f"Epoch {epoch + 1:03d} | "
        f"Mean Val MAE={mean_val_mae:.6f} | "
        f"Patience={no_improve_count}/{PATIENCE}"
    )

    for name in probe_names:
        print(
            f"{name:<12} | "
            f"train MAE {train_mae[name]:.6f} | "
            f"val MAE {val_mae[name]:.6f} | "
            f"1-MAE {1.0 - val_mae[name]:.6f} | "
            f"best 1-MAE {1.0 - best_mae[name]:.6f}"
        )

    if no_improve_count >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch + 1}. "
            f"Best mean validation MAE={best_mean_val_mae:.6f}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary
# ============================================================

print("\n" + "=" * 90)
print("FINAL TIMESFORMER VIDEO-ONLY LINEAR PROBE RESULTS")
print("=" * 90)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'MAE':>14}"
    f"{'1-MAE':>14}"
)
print("-" * 90)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mae[name]:>14.6f}"
        f"{1.0 - best_mae[name]:>14.6f}"
    )

print("=" * 90)

del model
del backbone
del optimizer
del train_dataloader
del val_dataloader

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
Backbone trainable parameters: 0
Probe trainable parameters:    69953


Probe Train 1/50: 100%|█████████▉| 1497/1498 [02:09<00:00, 12.22it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Probe Valid 1/50: 100%|██████████| 498/498 [00:48<00:00, 10.26it/s]



------------------------------------------------------------------------------------------
Epoch 001 | Mean Val MAE=0.112494 | Patience=0/10
block1       | train MAE 0.078240 | val MAE 0.114558 | 1-MAE 0.885442 | best 1-MAE 0.885442
block2       | train MAE 0.077896 | val MAE 0.114276 | 1-MAE 0.885724 | best 1-MAE 0.885724
block3       | train MAE 0.077178 | val MAE 0.106179 | 1-MAE 0.893821 | best 1-MAE 0.893821
block4       | train MAE 0.076366 | val MAE 0.110888 | 1-MAE 0.889112 | best 1-MAE 0.889112
block5       | train MAE 0.077169 | val MAE 0.118204 | 1-MAE 0.881796 | best 1-MAE 0.881796
block6       | train MAE 0.076803 | val MAE 0.114685 | 1-MAE 0.885315 | best 1-MAE 0.885315
block7       | train MAE 0.077055 | val MAE 0.104388 | 1-MAE 0.895612 | best 1-MAE 0.895612
block8       | train MAE 0.078239 | val MAE 0.111032 | 1-MAE 0.888968 | best 1-MAE 0.888968
block9       | train MAE 0.078206 | val MAE 0.106850 | 1-MAE 0.893150 | best 1-MAE 0.893150
block10      | train MAE 0.077

Probe Valid 2/50: 100%|██████████| 498/498 [00:48<00:00, 10.17it/s]



------------------------------------------------------------------------------------------
Epoch 002 | Mean Val MAE=0.116053 | Patience=1/10
block1       | train MAE 0.069718 | val MAE 0.110055 | 1-MAE 0.889945 | best 1-MAE 0.889945
block2       | train MAE 0.070928 | val MAE 0.121299 | 1-MAE 0.878701 | best 1-MAE 0.885724
block3       | train MAE 0.070828 | val MAE 0.116489 | 1-MAE 0.883511 | best 1-MAE 0.893821
block4       | train MAE 0.072496 | val MAE 0.127087 | 1-MAE 0.872913 | best 1-MAE 0.889112
block5       | train MAE 0.069895 | val MAE 0.105321 | 1-MAE 0.894679 | best 1-MAE 0.894679
block6       | train MAE 0.071331 | val MAE 0.119299 | 1-MAE 0.880701 | best 1-MAE 0.885315
block7       | train MAE 0.070175 | val MAE 0.115072 | 1-MAE 0.884928 | best 1-MAE 0.895612
block8       | train MAE 0.071293 | val MAE 0.117650 | 1-MAE 0.882350 | best 1-MAE 0.888968
block9       | train MAE 0.070444 | val MAE 0.114845 | 1-MAE 0.885155 | best 1-MAE 0.893150
block10      | train MAE 0.071

...


------------------------------------------------------------------------------------------
Epoch 035 | Mean Val MAE=0.101997 | Patience=8/10
block1       | train MAE 0.049572 | val MAE 0.101200 | 1-MAE 0.898800 | best 1-MAE 0.899008
block2       | train MAE 0.049738 | val MAE 0.101959 | 1-MAE 0.898041 | best 1-MAE 0.898288
block3       | train MAE 0.049742 | val MAE 0.102227 | 1-MAE 0.897773 | best 1-MAE 0.898346
block4       | train MAE 0.049874 | val MAE 0.102131 | 1-MAE 0.897869 | best 1-MAE 0.898369
block5       | train MAE 0.049750 | val MAE 0.102179 | 1-MAE 0.897821 | best 1-MAE 0.898748
block6       | train MAE 0.049803 | val MAE 0.102171 | 1-MAE 0.897829 | best 1-MAE 0.898698
block7       | train MAE 0.049822 | val MAE 0.101854 | 1-MAE 0.898146 | best 1-MAE 0.898634
block8       | train MAE 0.049848 | val MAE 0.101909 | 1-MAE 0.898091 | best 1-MAE 0.898720
block9       | train MAE 0.049903 | val MAE 0.102127 | 1-MAE 0.897873 | best 1-MAE 0.898456
block10      | train MAE 0.049

Probe Valid 36/50: 100%|██████████| 498/498 [00:49<00:00, 10.13it/s]



------------------------------------------------------------------------------------------
Epoch 036 | Mean Val MAE=0.102058 | Patience=9/10
block1       | train MAE 0.049342 | val MAE 0.101926 | 1-MAE 0.898074 | best 1-MAE 0.899008
block2       | train MAE 0.049620 | val MAE 0.101868 | 1-MAE 0.898132 | best 1-MAE 0.898288
block3       | train MAE 0.049582 | val MAE 0.102160 | 1-MAE 0.897840 | best 1-MAE 0.898346
block4       | train MAE 0.049588 | val MAE 0.102084 | 1-MAE 0.897916 | best 1-MAE 0.898369
block5       | train MAE 0.049456 | val MAE 0.101919 | 1-MAE 0.898081 | best 1-MAE 0.898748
block6       | train MAE 0.049616 | val MAE 0.101711 | 1-MAE 0.898289 | best 1-MAE 0.898698
block7       | train MAE 0.049681 | val MAE 0.101786 | 1-MAE 0.898214 | best 1-MAE 0.898634
block8       | train MAE 0.049660 | val MAE 0.102114 | 1-MAE 0.897886 | best 1-MAE 0.898720
block9       | train MAE 0.049687 | val MAE 0.101948 | 1-MAE 0.898052 | best 1-MAE 0.898456
block10      | train MAE 0.049

Probe Valid 37/50: 100%|██████████| 498/498 [00:49<00:00, 10.09it/s]


------------------------------------------------------------------------------------------
Epoch 037 | Mean Val MAE=0.102329 | Patience=10/10
block1       | train MAE 0.049304 | val MAE 0.101519 | 1-MAE 0.898481 | best 1-MAE 0.899008
block2       | train MAE 0.049561 | val MAE 0.102551 | 1-MAE 0.897449 | best 1-MAE 0.898288
block3       | train MAE 0.049544 | val MAE 0.102286 | 1-MAE 0.897714 | best 1-MAE 0.898346
block4       | train MAE 0.049709 | val MAE 0.102416 | 1-MAE 0.897584 | best 1-MAE 0.898369
block5       | train MAE 0.049564 | val MAE 0.102410 | 1-MAE 0.897590 | best 1-MAE 0.898748
block6       | train MAE 0.049624 | val MAE 0.102224 | 1-MAE 0.897776 | best 1-MAE 0.898698
block7       | train MAE 0.049623 | val MAE 0.102349 | 1-MAE 0.897651 | best 1-MAE 0.898634
block8       | train MAE 0.049657 | val MAE 0.102349 | 1-MAE 0.897651 | best 1-MAE 0.898720
block9       | train MAE 0.049654 | val MAE 0.102381 | 1-MAE 0.897619 | best 1-MAE 0.898456
block10      | train MAE 0.04